<img src="../images/cads-logo.png" style="height: 100px;" align=left>  <img src="../images/NLP.jpeg" style="height: 200px;" align=right, width="300">
# Natural Language Processing
### Day 3

# Contents:

- [Sentiment Analysis](#Sentiment-Analysis)
    - [Case Study: Reviews of Phones from Amazon](#Case-Study:-Reviews-of-Phones-from-Amazon)
        - [Dataset Content](#Dataset-Content)
    - [Loading Libraries and Data](#Loading-Libraries-and-Data)
    - [Model Selection in scikit-learn](#Model-Selection-in-scikit-learn)
    - [Extracting Features from Text Files](#Extracting-Features-from-Text-Files)
        - [Bag-of-words (BOW)](#Bag-of-words-(BOW))
            - [Transform Words into Vectors (CountVectorizer)](#Transform-Words-into-Vectors-(CountVectorizer))
            - [Exercise A](#Exercise-A)
        - [Logistic Regression Classification](#Logistic-Regression-Classification)
        - [Term Frequency Inverse Document Frequency (tfidf)](#Term-Frequency-Inverse-Document-Frequency-(tfidf))
            - [Exercise B](#Exercise-B)
        - [n-grams](#n-grams)
    - [Visualize top N uni-grams, bi-grams & tri-grams](#Visualize-top-N-uni-grams,-bi-grams-&-tri-grams)
        - [Exercise C](#Exercise-C)
- [Topic Modeling](#Topic-Modeling)
    - [What is Topic Modeling?](#What-is-Topic-Modeling?)
    - [Latent Dirichlet Allocation (LDA)](#Latent-Dirichlet-Allocation-(LDA))
    - [LDA for Topic Modeling](#LDA-for-Topic-Modeling)
        - [Case Study: Reviews of Foods from Amazon](#Case-Study:-Reviews-of-Foods-from-Amazon)
        - [Find 10 Words with the Highest Probability for the First Topic](#Find-10-Words-with-the-Highest-Probability-for-the-First-Topic)
    - [Visualize the Topic Modeling - pyLDAvis Package](#Visualize-the-Topic-Modeling---pyLDAvis-Package)
        - [Data Pre-processing](#Data-Pre-processing)
        - [Building Dictionary](#Building-Dictionary)
        - [Building Corpus](#Building-Corpus)
        - [Building LDA Model](#Building-LDA-Model)
    - [Visualizing the Topics](#Visualizing-the-Topics)
        - [How to infer pyLDAvis’s output?](#How-to-infer-pyLDAvis’s-output?)
    - [Word Clouds of Top N Keywords in Each Topic](#-Word-Clouds-of-Top-N-Keywords-in-Each-Topic)
    - [Plotting Words Importance](#Plotting-Words-Importance)
- [Assignment 1](#Assignment-1)
- [Assignment 2](#Assignment-2)

# Sentiment Analysis

## Case Study: Reviews of Phones from Amazon

What better way than to use a popular use case application: *Amazon review sentiment analysis* to better understand how text information can be parsed and processed into something useful for Machine Learning.

We will be working on a large dataset of reviews of unlocked mobile phones sold on Amazon.com that has been collected by Crawlers et al. in December, 2016. The Amazon reviews dataset consists of 400 thousand reviews to find out insights with respect to reviews, ratings, price and their relationships.

### Dataset Content 

The fields in the dataset are given as follows:

**Product Name**
* Type: String
* Description: Name of the mobile phone

**Brand Name**
* Type: String
* Description: Brand of the mobile phone

**Price**
* Type: Float
* Description: Price of the mobile phone

**Rating**
* Type: Integer
* Description: Rating of the mobile phone given by the user (1-5)

**Review**
* Type: String
* Description: Reviews of the mobile phone given by the user

**Review Votes**
* Type: Integer
* Description: Number of people who have found the review helpful

Our main end goal here is to learn to extract meaningful information from a subset of these reviews to build a machine learning model that can predict whether a certain reviewer liked or disliked a mobile phones.

## Loading Libraries and Data

In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
# Colab setup
!pip install gensim pyldavis wordcloud
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/TM NLP/TM NLP Day 3/data/'

In [ ]:
# Read in the data
df = pd.read_csv(DATA + 'Amazon/Amazon_Unlocked_Mobile.csv.zip', encoding="utf8")

# Shuffle rows of dataframe
df = df.sample(frac=1, random_state=10)

In [ ]:
df.head()

In [ ]:
len(df)

In [ ]:
# Drop missing values
df.dropna(inplace=True)

# Remove any 'neutral' ratings equal to 3
df = df[df['Rating'] != 3]

# Encode 4s and 5s as 1 (rated positively)
# Encode 1s and 2s as 0 (rated poorly)
df['Positively Rated'] = np.where(df['Rating'] > 3, 1, 0)

We would like to remove certain noises from the **Reviews** column of the data, such as numbers, punctuations and non-ASCII characters. We would also want to remove certain stop words from the data. We do not remove all stop words as some stop words such as *"very", "some", "more", "no", "but", etc* are important in this case study.

In [ ]:
# Pre-processing
custom_stopwords = ["i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours", "yourself", "yourselves", "he", "him", "his", "himself", "she", "her", "hers", "herself", "it", "its", "itself", "they", "them", "their", "theirs", "themselves", "what", "which", "who", "whom", "this", "that", "these", "those", "am", "is", "are", "was", "were", "be", "been", "being", "have", "has", "had", "having", "do", "does", "did", "doing", "a", "an", "the", "and", "but", "if", "or", "because", "as", "until", "while", "of", "at", "by", "for", "with", "about", "against", "between", "into", "through", "during", "before", "after", "above", "below", "to", "from", "up", "down", "in", "out", "on", "off", "over", "under", "again", "further", "then", "once", "here", "there", "when", "where", "why", "how", "any", "each", "few", "other", "such", "only", "own", "same", "than", "s", "t", "can", "will", "just", "should", "now"]

# Remove punctuations
df['Reviews'] = [re.sub(r'[!"#$%&()*+,-./:;<=>?@[\]^_`{|}~]', '', text) for text in df['Reviews']]
# Remove numbers
df['Reviews'] = [re.sub(r'[0123456789]', '', text) for text in df['Reviews']]
# Remove non-ASCII characters
df['Reviews'] = [re.sub(r'[^\x00-\x7f]', '', text) for text in df['Reviews']]
# Remove certain stop words
df["Reviews"] = df['Reviews'].apply(lambda x: " ".join([word for word in x.lower().split() if word not in custom_stopwords]))
df['Reviews'] = [re.sub(r'[^\x00-\x7F]+', '', text) for text in df['Reviews']]
# Remove custom stopwords
df['Reviews'] = [' '.join(w for w in text.split() if w.lower() not in custom_stopwords) for text in df['Reviews']]

In [ ]:
df.head()

## Model Selection in scikit-learn

As we are interested in building a machine learning model, it is important to split the dataset into training and testing dataset so that we can evaluate our model. Training the model on the whole dataset will lead to overfitting as the model would not perform well on unseen data.

We can split a dataset into training and testing set using the `train_test_split` function of the `sklearn.model_selection` module. We give a `random_state` value as the seed such that the training and testing set are always split in the same way whenever we run the code.

In [ ]:
from sklearn.model_selection import train_test_split

# Split data into train and test subsets
X_train, X_test, y_train, y_test = train_test_split(df['Reviews'], 
                                                    df['Positively Rated'], 
                                                    random_state=0)

In [ ]:
print( X_train.iloc[10])   

In [ ]:
# The shape of the training data
X_train.shape

In [ ]:
# The shape of the testing data
X_test.shape

# Extracting Features from Text Files

Text files usually consist of a series of words (ordered). In order to run machine learning algorithms, we need to convert the text files into numerical feature vectors. We can do this by using the bag of words model.

## Bag-of-words (BOW)

Bag-of-words (BOW) model allows us to represent text as numerical feature vectors. The idea behind BOW is quite simple and can be summarized as follows:

    1) Create a vocabulary of unique tokens (or words) from the entire set of documents.
    2) Construct a feature vector from each document that contains the counts of how often each word occurs in the particular document.

Since the unique words in each document represent only a small subset of all the words in the bag-of-words vocabulary, the feature vectors will consist of mostly zeros, in which we call them sparse. For this reason we say that bags of words are typically **high-dimensional sparse datasets**.

We first segment each text file into words (for English, we split by space) and count the number of times each word occurs in each document. Then we can assign to each word an integer id. Each unique word in our dictionary will correspond to a feature (descriptive feature).

### Transform Words into Vectors (CountVectorizer)

To construct a bag-of-words model based on the word counts in the respective documents, we can use the `CountVectorizer` class implemented in `scikit-learn`. As we will see in the following codes, the `CountVectorizer` class takes an array of text data, which can be documents or just sentences, and constructs the bag-of-words model for us.

scikit-learn has a high level component which will automatically create the feature vectors for us.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

docs = np.array([
    'The sun is shining',
    'The weather is sweet',
    'The sun is shining and the weather is sweet'])

# Fit the CountVectorizer to the training data 
vect1 = CountVectorizer().fit(docs)

# Transform the documents in the training data to a document-term matrix. 
bag = vect1.transform(docs)

We can use the `vocabulary_` attribute to get the vocabulary of the CounterVectorizer, which returns a dictionary matching the words to their counts in the text.

In [ ]:
vect1.vocabulary_

The `get_feature_names()` method returns the an array mapping the feature indices to feature names (words existing in the text).

For example, given the feature names as below, the word *'and'* has the feature vector `[1, 0, 0, 0, 0, 0, 0]` and the word *'sun'* has the feature vector `[0, 0, 0, 1, 0, 0, 0]`.

In [ ]:
vect1.get_feature_names_out()

Once we have transformed the documents in the training data into a document-term matrix, we can use the `toarray()` method to convert the document into arrays of feature vectors of each sentence.

In [ ]:
bag.toarray()

In [ ]:
bag.shape

In [ ]:
bag

### <font color=green> Exercise A</font>

1) Perform CountVectorizer on the training data, and save the document-term matrix into the variable `X_train_vectorized`.

2) Determine: 
    - The number of features 
    - The shape of sparse matrix

3) Save the feature-matrix in a dataframe.

In [ ]:
# 1)
vect = CountVectorizer().fit(X_train)
X_train_vectorized = vect.transform(X_train)

In [ ]:
# 2)
print(len(vect.get_feature_names_out()))
print(X_train_vectorized.shape)

In [ ]:
# 3) 
X_train_df = pd.DataFrame.sparse.from_spmatrix(X_train_vectorized, columns=vect.get_feature_names_out())
X_train_df.head()

### Logistic Regression Classification

We will train a logistic regression model to classify the Amazon reviews into positive and negative reviews by using the feature matrix. 

In [ ]:
from sklearn.linear_model import LogisticRegression

# Train the model
model = LogisticRegression()
model.fit(X_train_vectorized, y_train)

In [ ]:
from sklearn.metrics import roc_auc_score

# Predict the transformed test documents
predictions = model.predict(vect.transform(X_test))
y_proba = model.predict_proba(vect.transform(X_test))
                              
print('AUC:', roc_auc_score(y_test, y_proba[:,1])) 

We can use the `.coef_` attribute to get the coefficients of the regression model. The regression coefficient describes the size and direction of the relationship between a predictor and the response variable. Coefficients are the numbers by which the values of the term are multiplied in a regression equation.

In [ ]:
print(model.coef_.shape)
model.coef_

In [ ]:
model.coef_[0].argsort()

We are interested to find the words with the smallest and largest coefficients. As seen above, we can sort the coefficients of the model. The `argsort()` method sort the coefficients by value but returns the sorted indices of the original array. We are not interested in the actual values of the coefficients, but the words associated to the coefficients.

To obtain the words, we can use Python's slicing method. Slicing the `feature_names` array with the indices of the smallest or largest coefficients will return the feature names (words).

In [ ]:
# Get the feature names as numpy array
feature_names = np.array(vect.get_feature_names_out())

# Sort the coefficients from the model
sorted_coef_index = model.coef_[0].argsort()

# Find the 10 smallest and 10 largest coefficients
print('Smallest Coefs:' )
print(feature_names[sorted_coef_index[:10]])
      
print('\n Largest Coefs:')      
print(feature_names[sorted_coef_index[:-11:-1]])

## Term Frequency Inverse Document Frequency (tfidf)

When we are analyzing text data, we often encounter words that occur across multiple documents for both classes. Those frequently occurring words typically do not contain useful or discriminatory information. In this section, we will learn about a useful technique called **term frequency-inverse document frequency** (*tf-idf*) that can be used to downweight those frequently occurring words in the feature vectors. In other words, we can use tf-idf to reduce the weightage of more common words like (the, is, an etc.) which occurs in all document.

The *tf-idf* can be defined as the product of the term frequency and the inverse document frequency:

\begin{align}
    \textit{tf-idf}(t,d) = \textit{tf}(t,d) \times \textit{idf}(t,d)
\end{align}

Here the <font color=green><b>tf(t,d)</b></font> is the term frequency that equal to **the count of words / total words in each document**. The inverse document frequency <font color=green><b>idf(t,d)</b></font> can be calculated as:

\begin{align}
    \textit{idf}(t,d) = log\frac{n_d}{\textit{df}(d,t)}
\end{align}

where <font color=green><b>$n_d$</b></font> is **the total number of documents**, and <font color=green><b>df(d,t)</b></font> is **the number of documents *d* that contain the term *t***. The *log* is used to ensure that low document frequencies are not given too much weight.


scikit-learn implements yet another vectorizer, the `TfidfVectorizer`, to create feature vectors as tf-idfs.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

docs = np.array([
    'The sun is shining',
    'The weather is sweet',
    'The sun is shining and the weather is sweet'])

vect2 = TfidfVectorizer().fit(docs)
bag2 = vect2.transform(docs)
bag2.toarray()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Fit the TfidfVectorizer to the training data 
vect = TfidfVectorizer(min_df=5).fit(X_train)

In [ ]:
len(vect.get_feature_names_out())

In [ ]:
X_train_vectorized = vect.transform(X_train)

# Train the model
model = LogisticRegression()
model.fit(X_train_vectorized, y_train)

# Model prediction
predictions = model.predict(vect.transform(X_test))
y_proba = model.predict_proba(vect.transform(X_test))
                              
print('AUC: ', roc_auc_score(y_test, y_proba[:,1])) 

### <font color=green> Exercise B</font> 

1) Predict the two reviews below as either *negative* or *positive* using our model: 

        ['no an issue, phone is working',
         'an issue, phone is not working']       

In [ ]:
model.predict(vect.transform(['no an issue, phone is working',
                              'an issue, phone is not working']))

## n-grams

The sequence of items in the bag-of-words model that we just created is also called the **1-gram** or **unigram model** — each item or token in the vocabulary represents a single word. Generally, *the contiguous sequences of items in NLP* — words, letters, or symbols— is also called an n-gram. 

The choice of the number *n* in the n-gram model depends on the particular application. For instance, spam filtering applications tend to use *n=3* or *n=4* for good performances.

To summarize the concept of the n-gram representation, the 1-gram and 2-gram representations of our first document "the sun is shining" would be constructed as follows:
    - 1-gram: "the", "sun", "is", "shining"
    - 2-gram: "the sun", "sun is", "is shining"

The `CountVectorizer` class in scikit-learn allows us to use different n-gram models via its `ngram_range` parameter. By default, it uses a 1-gram representation.

In [ ]:
# Try 2-gram representation
docs = np.array([
    'The sun is shining',
    'The weather is sweet',
    'The sun is shining and the weather is sweet'])

vect3=CountVectorizer(ngram_range=(1,2)).fit(docs)
bag3=vect3.transform(docs)

In [ ]:
vect3.vocabulary_

In [ ]:
# Get the 2-gram representation
vect = CountVectorizer(min_df=5, ngram_range=(1,2)).fit(X_train)

X_train_vectorized = vect.transform(X_train)

In [ ]:
len(vect.get_feature_names_out())

In [ ]:
# Train the model
model = LogisticRegression()
model.fit(X_train_vectorized, y_train)

# Model prediction
test_predictions = model.predict(vect.transform(X_test))
y_proba = model.predict_proba(vect.transform(X_test))
                              
print('AUC: ', roc_auc_score(y_test, y_proba[:,1])) 

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, recall_score

# Get the f1 score of the model
f1_score(y_test,test_predictions,average='micro')

In [ ]:
# Get the accuracy score of the model
accuracy_score(y_test,test_predictions)

In [ ]:
# Get the recall score of the model
recall_score(y_test,test_predictions,average='micro')

In [ ]:
feature_names = np.array(vect.get_feature_names_out())

sorted_coef_index = model.coef_[0].argsort()

print('Smallest Coefs:' )
print(feature_names[sorted_coef_index[:10]])
      
print('\n Largest Coefs:')      
print(feature_names[sorted_coef_index[:-11:-1]])

In [ ]:
print(model.predict(vect.transform(['no an issue, phone is working',
                                    'an issue, phone is not working'])))

These reviews are now correctly identified.

### Visualize top N uni-grams, bi-grams & tri-grams

We can use the `CountVectorizer` to visualize the top 20 unigrams, bi-grams and tri-grams.

We need a function to extract the top n words (unigrams, bi-grams or tri-grams) from a corpus. We first look at the general function to obtain the top unigrams using the following function `get_top_n_words()`, which will work for any corpus. The function takes any `corpus` as input, and the top `n` words to obtain (default is 10). The corpus will then be fitted into the `CountVectorizer` (as a unigram in this case) to obtain the bag of words. We can then get the number of occurences of each word in the corpus. To get the top `n` words, we sort the words by their number of occurences and perform indices slicing to obtain the last `n` ones.

In [ ]:
# Most frequently occuring words
def get_top_n_words(corpus, n=10):
    bag_of_words = CountVectorizer().fit(corpus)
    doc_term_matrix = bag_of_words.transform(corpus)
    sum_words = doc_term_matrix.sum(axis=0)   # each column
    words_freq = [(word, sum_words[0, idx]) for word, idx in bag_of_words.vocabulary_.items()]
    words_freq =sorted(words_freq, key = lambda x: x[1], reverse=True)
    return words_freq[:n]

Now that we have a function to obtain the top occuring words in the corpus, we can visualize them. We feed our *Reviews* dataset into the function and obtain the top words as a dataframe. We can then use the `seaborn` module of Python to plot a dataframe. As this a frequency dataset, the `sns.barplot` is the most suitable chart to visualize this data.

In [ ]:
# Convert most freq words to dataframe for plotting bar plot
top_words = get_top_n_words(df['Reviews'], n=5)
top_df = pd.DataFrame(top_words, columns=["Word", "Freq"])

In [ ]:
top_df.head()

In [ ]:
import seaborn as sns

# Barplot of most freq words
sns.set(rc={'figure.figsize':(13,8)})
ax = sns.barplot(x="Word", y="Freq", data=top_df)
ax.set_title('Bar plot of most frequently occurring uni-grams')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30)

### <font color=green> Exercise C</font> 

1) Write a code to use the `CountVectoriser` to visualise the top 20 bi-grams and tri-grams.

*Hint: you can use `max_features` parameter for CountVectorizer to limit the CountVectorizer to build a vocabulary that only consider the **top max_features** ordered by term frequency across the corpus.*

In [ ]:
# The most freq Bi-grams
def get_top_n_grams(corpus, ng, n=20):
    v = CountVectorizer(ngram_range=(ng, ng), max_features=n).fit(corpus)
    s = v.transform(corpus).sum(axis=0)
    return sorted([(w, s[0, i]) for w, i in v.vocabulary_.items()], key=lambda x: x[1], reverse=True)

top_df = pd.DataFrame(get_top_n_grams(df['Reviews'], 2), columns=['Bi-gram', 'Freq'])
ax = sns.barplot(x='Bi-gram', y='Freq', data=top_df)
ax.set_title('Top 20 bi-grams')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)

In [ ]:
# The most freq Tri-grams
top_df = pd.DataFrame(get_top_n_grams(df['Reviews'], 3), columns=['Tri-gram', 'Freq'])
ax = sns.barplot(x='Tri-gram', y='Freq', data=top_df)
ax.set_title('Top 20 tri-grams')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)

# Topic Modeling

## What is Topic Modeling?

A recurring subject in NLP is to understand large corpus of texts through topics extraction. Whether to analyze users’ online reviews, products’ descriptions, or text entered in search bars, understanding key topics will always come in handy.

Topic modeling is an unsupervised technique that intends to analyze large volumes of text data by clustering the documents into groups. In the case of topic modeling, the text data do not have any labels attached to it. Rather, topic modeling tries to group the documents into clusters based on similar characteristics as we see in the image below. Cluster a large number of newspaper articles that belong to the same category is an example of topic modeling.

<img src="../images/LDA1.png" style="height: 300px;" >

Two approaches are mainly used for topic modeling:<br>
1) Latent Dirichlet Allocation or LDA in short<br>
2) Non-Negative Matrix factorization or NMF or NNMF in short

In this section, we will review LDA approaches and see how it can be applied to topic modeling in Python.

## Latent Dirichlet Allocation (LDA)

Latent Dirichlet Allocation (LDA) is based upon two general assumptions:

- Documents that have similar words usually have the same topic
- Documents that have groups of words frequently occurring together usually have the same topic

LDA considers each document as a collection of topics in a certain proportion. And each topic as a collection of keywords, again, in a certain proportion. A topic is nothing but a collection of dominant keywords that are typical representatives. Just by looking at the keywords, we can identify what the topic is all about.

LDA takes documents as input and returns the found topics as output. The model also returns in what percentage each document talks about each topic as shown in the image below.

<img src="../images/LDA2.png" style="height: 220px;">

It is important to mention here that it is extremely difficult to evaluate the performance of topic modeling since there are no right answers.

## LDA for Topic Modeling

In this section we will see how Python can be used to implement LDA for topic modeling. 

### Case Study: Reviews of Foods from Amazon

This dataset consists of reviews of fine foods from Amazon for different products in the food category. The dataset contains around 500k reviews up to October 2012. Reviews include product and user information, ratings, and a plain text review. However, due to memory constraints, we will perform LDA only on the first 20k records. Furthermore, we will group the user reviews into 5 categories. 

The dataset can be downloaded from Kaggle, available [here](https://www.kaggle.com/sdxingaijing/topic-model-lda-algorithm/data).

The first step is to import the dataset along with the required libraries.

In [ ]:
import gensim
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [ ]:
reviews_datasets = pd.read_csv(DATA + 'Amazon/Reviews.csv.zip')
reviews_datasets = reviews_datasets.head(20000)
# Remove null values from the data set
reviews_datasets.dropna()
# Peek the data
reviews_datasets.head()

In [ ]:
# Let's look at review number 102
reviews_datasets['Text'][102]   

Before we can apply LDA, we need to create vocabulary of all the words in our data. We could do it with the help of a `CountVectorizer` class and then create a document-term matrix. 

We can give `max_df=0.8` as a parameter in `CountVectorizer` to include only those words that appear in less than 80% of the document and appear in at least 2 documents. We can also give `stop_words='english'` to exclude all stop words.

We specify the values of the **Text** column in the data set as unicode characters using the attribute `astype('U')`.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

count_vect = CountVectorizer(max_df=0.8, min_df=2, 
                             stop_words='english')

doc_term_matrix = count_vect.fit_transform(reviews_datasets["Text"].values.astype('U'))

In [ ]:
# Let's look at our document term matrix:
doc_term_matrix

Each of the 20k documents is represented as a 14546 dimensional vector. This means that our vocabulary has 14546 words.

Next, we will use LDA to create topics along with the probability distribution for each word in our vocabulary for each topic.

We use the `LatentDirichletAllocation` class from the `sklearn.decomposition` library to perform LDA on our document-term matrix. The parameter `n_components` specifies the number of categories, or topics, that we want our text to be divided into.

We fit the LDA model with the document-term matrix as the training data.

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

LDA = LatentDirichletAllocation(n_components=5, random_state=42)
LDA.fit(doc_term_matrix)

Let's randomly fetch 10 words from our vocabulary. We know that the count vectorizer contains all the words in our vocabulary. We can use the `get_feature_names()` method and pass it the ID of the word that we want to fetch.

In [ ]:
# Fetch 10 words from our vocabulary randomly
import random
                               
for i in range(10):
    random_id = random.randint(0,len(count_vect.get_feature_names_out())-1)
    print(count_vect.get_feature_names_out()[random_id])

### Find 10 Words with the Highest Probability for the First Topic

To get the first topic, we can use the `components_` attribute and pass a 0 index as the value. The `components_` of LDA are the variational parameters for topic word distribution.

In [ ]:
first_topic = LDA.components_[0]
first_topic

The first component contains the probabilities of 14546 words for topic 1. We can use the `argsort()` function to sort the indices according to probability values. Once sorted, the 10 words with the highest probabilities will now belong to the last 10 indices of the array. 

In [ ]:
# The indices of the 10 words with the highest probabilities
top_topic_words = first_topic.argsort()[-10:]
top_topic_words

These indices can then be used to retrieve actual words from the `count_vect` object.

In [ ]:
for i in top_topic_words:
    print(count_vect.get_feature_names_out()[i])

The words show that the first topic might be about **tea**.

Let's print the 10 words with highest probabilities for all the five topics.

In [ ]:
for i,topic in enumerate(LDA.components_):
    print(f'Top 10 words for topic #{i+1}:')
    print([count_vect.get_feature_names_out()[i] for i in topic.argsort()[-10:]])
    print('\n')

The output shows that the second topic might contain reviews about **snacks** (chips/chocolates). Similarly, the third topic might again contain reviews about **cold beverages** (soda/juices). We can see that there a few common words in all the categories becuase these words are used for almost all the topics. For instance "good", "great", "like" etc.

As a final step, we will add a column to the original data frame that will store the topic for the text.

To do so, we can use the `LDA.transform()` method and give as input our document-term matrix. This method will assign the probability of all the topics to each document.

In [ ]:
topic_values = LDA.transform(doc_term_matrix)
print(topic_values.shape)
topic_values[1]

In the output, we hve an array of shape `(20000, 5)` which means that each of the document has 5 columns where each column corresponds to the probability value of the particular topic. 

To find the topic index with maximum value, we can call the `argmax()` method.

In [ ]:
reviews_datasets['Topic'] = topic_values.argmax(axis=1)
reviews_datasets.head()

We can see that there is a new column for the topic in the data frame.

## Visualize the Topic Modeling - pyLDAvis Package

We can use the `pyLDAvis` package to interpret the topics in a topic model that has been fit into a corpus of text data. The package extracts information from a fitted LDA topic model to inform an interactive web-based visualization. In addition, it is designed to work well with Jupyter notebooks and can also be saved to a stand-alone HTML file for easy sharing.

This package works very well with the `gensim` package in the case of topic modeling with LDA. So, we need to make our LDA model with `gensim`. The two main inputs of the LDA topic model are the `dictionary(id2word)` and the `corpus`. 

Before we create a LDA model, first we need to clean our dataset.

### Data Pre-processing

The data pre-processing procedures that we will perform here are:
- set all words to lower cases
- remove all punctuations
- remove certain stop words
- lemmatization

Note: we only want to remove certain stop words, as some stop words such as *"very", "some", "more", "no", "but", etc* are important in this case study.

In [ ]:
import nltk
nltk.download('stopwords'); nltk.download('wordnet'); nltk.download('punkt'); nltk.download('punkt_tab')
from nltk.corpus import stopwords 
from nltk.stem.wordnet import WordNetLemmatizer
import string

stopwords = set(stopwords.words('english'))
punctuation = set(string.punctuation) 
lemmatize = WordNetLemmatizer()

def cleaning(review):
    one = " ".join([i for i in review.lower().split() if i not in stopwords])
    two = "".join(i for i in one if i not in punctuation)
    three = " ".join(lemmatize.lemmatize(i) for i in two.split())
    return three

reviews_datasets["Text_cleaned"] = [cleaning(doc) for doc in reviews_datasets['Text']]

In [ ]:
reviews_datasets.head()

Once we have finished the data cleaning, we will tokenize all the reviews in the dataset.

In [ ]:
import nltk
from nltk.tokenize import word_tokenize

data_tokens = [nltk.word_tokenize(t) for t in reviews_datasets["Text_cleaned"]]

### Building Dictionary

We would like to create a term dictionary from our dataset. We can create one using the `Dictionary` class from the `gensim.corpora.dictionary` module.
 
A `Dictionary` is a mapping between words and their integer ids. Given the list of words/tokens, it encapsulates the mapping between normalized words/tokens and their integer ids.

In [ ]:
from gensim.corpora.dictionary import Dictionary

dictionary = Dictionary(data_tokens)

### Building Corpus

We want to convert our list of documents (corpus) into a document-term matrix using the dictionary prepared above. We can do this using the `doc2bow()` function. Given the document as input, the function converts the document into the bag-of-words (BoW) format = list of (token_id, token_count) tuples.

In [ ]:
doc_term_matrix = [dictionary.doc2bow(doc) for doc in data_tokens]

### Building LDA Model

When we have the dictionary and the corpus, we can use the `LdaModel()` ckass from the `gensim.models` module to build our LDA model. 

A brief description of the parameters that are used in LDA model.

- `corpus`: The input is the bag of words vector.

- `id2word`: This requires the dictionary. it is Mapping from word IDs to words.

- `num_topics`: The number of latent topics that are to be extracted from the training corpus.

- `passes`: Number of passes through the corpus during training.

- `per_word_topics`: If True, the model also computes a list of topics, sorted in descending order of most likely topics for each word.

In [ ]:
from gensim.models import LdaModel

# Fitting LDA model on the document term matrix
lda_model = LdaModel(corpus = doc_term_matrix, 
                     id2word = dictionary,
                     num_topics=5,  
                     passes=50,
                     alpha='auto',
                     per_word_topics=True)

We can use the `show_topics()` method to get a representation for selected topics. The method can take in the following optional parameters:
- `num_topics`: the number of topics to be returned (ignore if the `num_topics` is already specified in the initialization of LdaModel)
- `num_words`: the number of words to be presented for each topic. These will the most relevant words (assigned the highest probability for each topic)
- `formatted`: if **True**, the topic representations will be formatted as strings. If **False**, they are returned as tuples of *(word, probability)*

In general, the function returns a list of topics, each represented either as a string (when `formatted` == **True**) or word-probability pairs.

In [ ]:
# Show Topics
from pprint import pprint

pprint(lda_model.show_topics(formatted=False))

### Visualizing the Topics

Now, we can finally get to visualizing the topics using `pyLDAvis`, a Python library for interactive topic model visualization. This is a port of the fabulous [R package](https://github.com/cpsievert/LDAvis) by Carson Sievert and Kenny Shirley.

In [ ]:
# Uncomment the following code to install the module
!pip install pyldavis

Let us import the `pyLDAvis.gensim` module from the package. Everytime we want to display the visualization, we would need to call the `display()` function. To avoid tediously typing in `display` all the time, we can run the following code:
```python
pyLDAvis.enable_notebook()
```

In [ ]:
import pyLDAvis
import pyLDAvis.gensim_models
pyLDAvis.enable_notebook()

We can use the `prepare()` function to transform our data into the format needed for the visualization. In the case of a `gensim` model, the function takes in as input the `lda`, `corpus` and `dictionary`.

In [ ]:
# Visualize the topics
lda_vis = pyLDAvis.gensim_models.prepare(lda_model, 
                                  doc_term_matrix,
                                  dictionary,
                                  mds='mmds')
lda_vis

We can use the `save_html` function to save the embedded visualization to a file. This will produce a self-contained HTML file. Internet access is still required for the D3 and LDAvis libraries.

In [ ]:
pyLDAvis.save_html(lda_vis, 'vis.html')

#### How to infer pyLDAvis’s output?

- It is an interactive plot, where each bubble represents a topic. 
- The larger the bubble size, the more prevalent is that topic.
- A good topic model will have large non-overlapping bubbles in the chart.
- A model with too many topics, will typically have many overlaps, small sized bubbles clustered in one region of the chart.
- The bar plot one right-hand side of the screenshot shows the frequency of the terms in the topic, out of the total term frequency in the documents.
- If you move the cursor over one of the bubbles, the words and bars on the right-hand side will update. These words are the salient keywords that form the selected topic.

##  Word Clouds of Top N Keywords in Each Topic

Now that we know what are the topic keywords in each topic, a wordcloud with the size of the words proportional to the weight is a pleasant sight.

A word cloud is a visualization wherein the most frequent words appear in large size and the less frequent words appear in smaller sizes.

We can create and display word clouds in Python using the `wordcloud` module. The `WordCloud()` object can be given arguments such as width, height and maximum font size of the word cloud. We set the `random_state` of the word cloud to a given number to ensure that we get the same word cloud everytime we run the code as the world cloud is randomized. To obtain the word cloud of a given `text`, we use the `generate(text)` method of `WordCloud()`. We can then display the word cloud using the `matplotlib`.

Let us visualize top N keywords in each topic using the wordcloud plot.

In [ ]:
# Wordcloud of Top N words in each topic
from wordcloud import WordCloud

cloud = WordCloud(background_color='white',   
                  width=2500,
                  height=1800,
                  max_words=10,
                  colormap='tab10', 
                  random_state=21)

topics = lda_model.show_topics(formatted=False)

fig, axes = plt.subplots(3, 2, figsize=(10,10), sharex=True, sharey=True)

for i, ax in enumerate(axes.flatten()):
    if i == 5:
        continue
    fig.add_subplot(ax)
    topic_words = dict(topics[i][1])   
    cloud.generate_from_frequencies(topic_words, max_font_size=300) 
    plt.gca().imshow(cloud)    
    plt.gca().set_title('Topic ' + str(i+1), fontdict=dict(size=16))
    plt.gca().axis('off')


plt.subplots_adjust(wspace=0, hspace=0)
plt.axis('off')
plt.margins(x=0, y=0)
plt.tight_layout()
plt.show()

## Plotting Words Importance 

Besides from word clouds, we can also plot the important words of each word using a simple bar chart. We display the top N keywords in each topic, sorting them by their importance in terms of their probability for that particular topic.

In [ ]:
fiz=plt.figure(figsize=(15,30))
for i in range(5):
    df=pd.DataFrame(lda_model.show_topic(i), columns=['term','prob']).set_index('term')
    
    plt.subplot(5,2,i+1)
    plt.title('Topic '+str(i+1))
    sns.barplot(x='prob', y=df.index, data=df, palette="ch:2.5,-.2") 
    plt.xlabel('probability')
    

plt.show()

# <font color=green>Assignment 1</font>

In this assignment, we will use the same English SMS messages dataset that we used in the previous assignment. Recall that the messages are tagged according to them being legitimate (**ham**) or **spam**. The dataset is slightly modifed for the assignment, with original one available [here](http://www.dt.fee.unicamp.br/~tiago/smsspamcollection/).

We will build several logistic regression models to classify the dataset, using different features for each model to compare their performances. The features that we will be using are as follows:
- Bag of Words
- tf-idf
- Bigrams
- Trigrams

We will compare the performance of each model by using the **f1 score** metric.

### Load the dataset

In [ ]:
import nltk
import re
import pandas as pd

# Load the dataset
f = open(DATA + 'sms_dataset.txt', 'r', encoding='utf-8')

# Get the data as a list of tokens
corpora = [re.split(r"\s", msg)[:-1] for msg in f]
# Get the labels of the data
labels = [sent.pop(0) for sent in corpora]
messages = [" ".join(sent).lower() for sent in corpora]

# Close the file
f.close()

In [ ]:
# Convert from text to dataframe
df = pd.DataFrame([messages, labels])
df = df.transpose()
df.columns = ["Message", "Label"]
df.head()

### Clean the data

Remove punctuations, numbers and non-ASCII characters from the **Messages**.

In [ ]:
df['Message'] = [re.sub(r'[^a-z\s]', '', m) for m in df['Message']]
df.head()

### Split data into training and testing

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df['Message'], df['Label'], random_state=0)

We extract a few examples from the test dataset to easily display the message and compare our predictions with the true label.

In [ ]:
test_ham_idx = []
test_spam_idx = []

for i, label in enumerate(y_test):
    if len(test_ham_idx) >= 5 and len(test_spam_idx) >= 5:
        break
    if label == 'ham' and len(test_ham_idx) < 5:
        test_ham_idx.append(i)
    if label == 'spam' and len(test_spam_idx) < 5:
        test_spam_idx.append(i)

test_ham_idx, test_spam_idx

Once you have train your model, obtain the predictions on the test dataset and run the function below with the predictions as input. The function will display 10 messages, their true labels and your predicted labels.

In [ ]:
def display_predictions(predictions):
    temp_msg = [list(X_test)[i] for i in test_ham_idx + test_spam_idx]
    temp_true = [list(y_test)[i] for i in test_ham_idx + test_spam_idx]
    temp_pred = [predictions[i] for i in test_ham_idx + test_spam_idx]
    temp_df = pd.DataFrame([temp_msg, temp_true, temp_pred])
    temp_df = temp_df.transpose()
    temp_df.columns = ["Message", "True Label", "Predicted Label"]
    display(temp_df)

### Bag of words

Build a logistic regression model using *bag of words*.

Print the F1 score and run the `display_predictions()` function on your predictions.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

vect = CountVectorizer().fit(X_train)
model = LogisticRegression().fit(vect.transform(X_train), y_train)
predictions = model.predict(vect.transform(X_test))
print('F1:', f1_score(y_test, predictions, pos_label='spam'))
display_predictions(predictions)

### tf-idf

Build a logistic regression model using *tf-idf*.

Print the F1 score and run the `display_predictions()` function on your predictions.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vect = TfidfVectorizer().fit(X_train)
model = LogisticRegression().fit(vect.transform(X_train), y_train)
predictions = model.predict(vect.transform(X_test))
print('F1:', f1_score(y_test, predictions, pos_label='spam'))
display_predictions(predictions)

### Bigrams

Build a logistic regression model using *bigrams*. Note: in the `ngram_range`, include both unigrams and bigrams.

Print the F1 score and run the `display_predictions()` function on your predictions.

In [ ]:
vect = CountVectorizer(ngram_range=(1,2)).fit(X_train)
model = LogisticRegression().fit(vect.transform(X_train), y_train)
predictions = model.predict(vect.transform(X_test))
print('F1:', f1_score(y_test, predictions, pos_label='spam'))
display_predictions(predictions)

### Trigrams

Build a logistic regression model using *trigrams*. Note: in the `ngram_range`, include unigrams, bigrams and trigrams.

Print the F1 score and run the `display_predictions()` function on your predictions.

In [ ]:
vect = CountVectorizer(ngram_range=(1,3)).fit(X_train)
model = LogisticRegression().fit(vect.transform(X_train), y_train)
predictions = model.predict(vect.transform(X_test))
print('F1:', f1_score(y_test, predictions, pos_label='spam'))
display_predictions(predictions)

# <font color=green>Assignment 2</font>

In this assignment, we will perform topic modelling on the same dataset (use the cleaned dataframe). Implement LDA on the dataset.

### Obtain the document-term matrix

Note: only include words that appear in less than 75% of the document and appear in at least 5 messages
Note: remove stopwords

In [ ]:
count_vect = CountVectorizer(max_df=0.75, min_df=5, stop_words='english')
doc_term_matrix = count_vect.fit_transform(df['Message'])
doc_term_matrix

### Perform LDA

Group the messages into **8** topics.

In [ ]:
# Use this seed for your random_state
seed = 42

from sklearn.decomposition import LatentDirichletAllocation
LDA = LatentDirichletAllocation(n_components=8, random_state=seed)
LDA.fit(doc_term_matrix)

### Print results

Print out the top **10** words in each topic.

In [ ]:
for i, topic in enumerate(LDA.components_):
    print(f'Top 10 words for topic #{i+1}:')
    print([count_vect.get_feature_names_out()[j] for j in topic.argsort()[-10:]])
    print()